In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [ ]:
import multiprocessing as mp

mp.set_start_method("spawn")

In [ ]:
import os
import sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# os.environ["MKL_THREADING_LAYER"] = "GNU"

sys.path.append("../02-encoding")

In [ ]:
### Shortcut for package import
from pkgimp import *

from nb2p import (
    codeop,
    fileop,
    token,
    database,
    dgraph,
    coderepr,
    config,
    npop,
    jsonencode,
)
from nb2p.notebook import Notebook
from nb2p.stmodel.dnn import BiLSTM, Transformer, DNNInput, Trainer

In [ ]:
TRAIN_DATASET_NAME = "distilkaggle"
TEST_DATASET_NAME = "distilkaggle"

In [ ]:
TRAIN_DIRS = config.dirs(dataset_name=TRAIN_DATASET_NAME)
TRAIN_DIRS.makedirs()

TEST_DIRS = config.dirs(dataset_name=TEST_DATASET_NAME)
TEST_DIRS.makedirs()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device
# device = torch.device("cpu")

## Load Data & Pre-processing

### Load dataset

In [ ]:
MAX_LENGTH = 256
MAX_LENGTH

Get the list of all samples by globbing the folder

In [ ]:
test_samples = glob.glob(str(TEST_DIRS.dfgtree / "test-*.lz4"))

In [ ]:
db, client = database.connect(dataset_name=TEST_DATASET_NAME, verbose=True)

Build List of X-y dicts

In [ ]:
ids = set(map(
    lambda x: str(x["_id"]),
    db.notebooksegments.find(
        {"prompted": True, "segment_ends.3": {"$exists": True}, "n_ast_children_of_segments": {"$lte": 256}},
        {"_id": 1},
    ),
))
len(ids)

Build List of X-y dicts

In [ ]:
def clean_segment_ends(segment_ends: List[int]):
    result = set()
    for x in segment_ends:
        if x >= MAX_LENGTH:
            return None, f"segment ends exceed max length: {x}"
        if x not in result and x >= 0:
            result.add(x)

    result = sorted(result)

    if len(result) == 0:
        return None, "empty segment ends after cleaning"

    return result, None


def build_shallow_dataset(samples: List[str]):
    for sample in tqdm(samples):
        # read data
        nb_id = sample.split("/")[-1].split(".")[-2].split("-")[-2]
        if nb_id not in ids:
            # print(f"WARN  ignore {sample}. Reason: should not be included")
            continue

        sample_dict = fileop.read_lz4(sample)
        sample_dict["segment_ends"], err = clean_segment_ends(
            sample_dict["segment_ends"]
        )
        if err:
            # print(f"WARN  ignore {sample}. Reason: {err}")
            continue

        sample_dict["y"] = npop.indices_to_binary(
            sample_dict["segment_ends"], sample_dict["segment_ends"][-1] + 1
        )

        func_encodings = []
        for fdef in sample_dict["func_defs"]:
            func_encodings.append(fdef["repr"])

        encodings = []
        for s in sample_dict["segments"]:
            encodings.extend([c.repr[0] for c in s["repr"].children])

        if len(encodings) == 0:
            # print(f"WARN  ignore {sample}. Reason: encodings is empty")
            continue

        sample_dict["x"] = np.array(encodings)

        code = "\n".join([s['code'] for s in sample_dict['segments']])

        del sample_dict["segments"]
        del sample_dict["func_defs"]
        
        sample_dict["gt_ast"] = sample_dict["segment_ends"]
        sample_dict['code'] = code

        yield sample_dict

In [ ]:
test_encodings = list(build_shallow_dataset(test_samples))

In [ ]:
len(test_encodings), test_encodings[0]

In [ ]:
TRAIN_SETUP_NAME = "astn4_256"
TEST_SETUP_NAME = (
    "astn4_256_cross" if TRAIN_DATASET_NAME != TEST_DATASET_NAME else TRAIN_SETUP_NAME
)
TRAIN_SETUP_NAME, TEST_SETUP_NAME

### Pad to max length

In [ ]:
display(test_encodings[0]["x"].shape, test_encodings[0]["x"].shape)
display(
    npop.pad(test_encodings[0]["x"], MAX_LENGTH).shape,
    npop.pad(test_encodings[0]["x"], MAX_LENGTH).shape,
)

In [ ]:
X_test_padded = np.zeros((len(test_encodings), MAX_LENGTH, 96), dtype=np.float32)
y_test_padded = np.zeros((len(test_encodings), MAX_LENGTH), dtype=np.float32)

In [ ]:
### NOTE: An interesting finding that applying multithreading is as expected on parallelism
### but not multiprocessing, as NumPy can only use single CPU core across processes.
from functools import partial

MAX_WORKERS = 32


def assign_data4pool(dest, data):
    i, arr = data
    if len(dest.shape) == 2:
        dest[i, : arr.shape[0]] = arr
    elif len(dest.shape) == 3:
        dest[i, : arr.shape[0], :] = arr


def mask_data4pool(data):
    return npop.mask(data["y"].shape[0], MAX_LENGTH)

In [ ]:
from functools import partial

X_test_padded_arr = thread_map(
    partial(assign_data4pool, X_test_padded),
    enumerate(d["x"] for d in test_encodings),
    max_workers=MAX_WORKERS,
)
y_test_padded_arr = thread_map(
    partial(assign_data4pool, y_test_padded),
    enumerate(d["y"] for d in test_encodings),
    max_workers=MAX_WORKERS,
)
print(len(X_test_padded_arr), len(y_test_padded_arr))

In [ ]:
test_mask = thread_map(mask_data4pool, test_encodings, max_workers=MAX_WORKERS)
print(test_mask[0])

In [ ]:
test_lengths = list(map(lambda x: (~x).sum(), test_mask))
print(len(test_lengths))
print(test_lengths[0])

In [ ]:
DATA_X = X_test_padded
DATA_Y = y_test_padded

In [ ]:
import gc
import ctypes

del gc.garbage[:]
gc.collect()
libc = ctypes.CDLL("libc.so.6")
libc.malloc_trim(0)

In [ ]:
CODE_X = [e['code'] for e in test_encodings]

## ST Models

### Decision Tree

In [ ]:
import nb2p.stmodel.dtree as dtree

config = dtree.DTreeConfig(max_depth=64)

_, segment_result = dtree.train_test(
    train_data=None,
    test_data=(DATA_X, DATA_Y),
    test_lengths=test_lengths,
    code=CODE_X,
    config=config,
    log_path=dtree.default_log_path(TEST_DIRS, TEST_SETUP_NAME, config),
    model_path=dtree.default_model_path(TRAIN_DIRS, TRAIN_SETUP_NAME, config),
    trained=True
)
segment_result[0]

In [ ]:
fileop.write_json(segment_result, TEST_DIRS.base / f"{TEST_SETUP_NAME}_dtree-segment-result.json")

In [ ]:
segment_result_dict = {i: s for i, s in enumerate(segment_result)}
print(next(iter(segment_result_dict.values())))

In [ ]:
from tqdm.contrib.concurrent import process_map

MAX_WORKERS = 32

ged_result = process_map(
    dgraph.do_compute_ged,
    segment_result_dict.items(),
    max_workers=MAX_WORKERS,
    chunksize=100,
)
len(ged_result)

In [ ]:
ged_result_valid = [g for g in ged_result if g is not None]
len(ged_result_valid)

In [ ]:
print(sum(ged_result_valid) / len(ged_result_valid))

### Random Forest

In [ ]:
import nb2p.stmodel.rforest as rforest

config = rforest.RForestConfig(max_depth=64)

_, segment_result = rforest.train_test(
    train_data=None,
    test_data=(DATA_X, DATA_Y),
    test_lengths=test_lengths,
    code=CODE_X,
    config=config,
    log_path=rforest.default_log_path(TEST_DIRS, TEST_SETUP_NAME, config),
    model_path=rforest.default_model_path(TRAIN_DIRS, TRAIN_SETUP_NAME, config),
    trained=True
)
segment_result[0]

In [ ]:
fileop.write_json(segment_result, TEST_DIRS.base / f"{TEST_SETUP_NAME}_rforest-segment-result.json")

In [ ]:
segment_result_dict = {i: s for i, s in enumerate(segment_result)}
print(next(iter(segment_result_dict.values())))

In [ ]:
from tqdm.contrib.concurrent import process_map

MAX_WORKERS = 32

ged_result = process_map(
    dgraph.do_compute_ged,
    segment_result_dict.items(),
    max_workers=MAX_WORKERS,
    chunksize=100,
)
len(ged_result)

In [ ]:
ged_result_valid = [g for g in ged_result if g is not None]
len(ged_result_valid)

In [ ]:
print(sum(ged_result_valid) / len(ged_result_valid))

### XGBoost

In [ ]:
import nb2p.stmodel.xgboost as xgboost

config = xgboost.XGBoostConfig(max_depth=8)

_, segment_result = xgboost.train_test(
    train_data=None,
    test_data=(DATA_X, DATA_Y),
    test_lengths=test_lengths,
    code=CODE_X,
    config=config,
    log_path=xgboost.default_log_path(TEST_DIRS, TEST_SETUP_NAME, config),
    model_path=xgboost.default_model_path(TRAIN_DIRS, TRAIN_SETUP_NAME, config),
    trained=True
)
segment_result[0]

In [ ]:
fileop.write_json(segment_result, TEST_DIRS.base / f"{TEST_SETUP_NAME}_xgboost-segment-result.json")

In [ ]:
segment_result_dict = {i: s for i, s in enumerate(segment_result)}
print(next(iter(segment_result_dict.values())))

In [ ]:
from tqdm.contrib.concurrent import process_map

MAX_WORKERS = 32

ged_result = process_map(
    dgraph.do_compute_ged,
    segment_result_dict.items(),
    max_workers=MAX_WORKERS,
    chunksize=100,
)
len(ged_result)

In [ ]:
ged_result_valid = [g for g in ged_result if g is not None]
len(ged_result_valid)

In [ ]:
print(sum(ged_result_valid) / len(ged_result_valid))

### Transformer

In [ ]:
MODEL_ABBR = "transformer"
MODEL_NAME = "transformer"

num_heads = 8
hidden_size = 512
num_layers = 4
epoch = 50

In [ ]:
# Instantiate the model
st_model = Transformer(
    input_size=MAX_LENGTH,
    hidden_size=hidden_size,
    num_heads=num_heads,
    num_layers=num_layers,
).to(device)

MODEL_PATH = (
    TRAIN_DIRS.log
    / f"{TRAIN_SETUP_NAME}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}-e{epoch}.pt"
)
print(MODEL_PATH)

st_model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device('cpu')))
st_model.eval()
print("Model loaded")

In [ ]:
segment_result = Trainer(
    st_model,
    num_epochs=300,
    batch_size=128,
    eval_freq=1,
    checkpoint_freq=1,
    learning_rate=1e-3,
    device=device,
    model_write_dir=TEST_DIRS.log,
    report_interval=50,
    window_size=2000,
    model_name=(
        f"{TEST_SETUP_NAME}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}"
    ),
    setup='bce',
).evaluate(
    DNNInput(X=X_test_padded, y=y_test_padded, code=CODE_X, mask=test_mask),
)

In [ ]:
segment_result_dict = {i: s for i, s in enumerate(segment_result)}
print(next(iter(segment_result_dict.values())))

In [ ]:
from tqdm.contrib.concurrent import process_map

MAX_WORKERS = 32

ged_result = process_map(
    dgraph.do_compute_ged,
    segment_result_dict.items(),
    max_workers=MAX_WORKERS,
    chunksize=1,
)
len(ged_result)

In [ ]:
ged_result_valid = [g for g in ged_result if g is not None]
len(ged_result_valid)

In [ ]:
print(sum(ged_result_valid) / len(ged_result_valid))

### NB2P w/o DTE

In [ ]:
MODEL_ABBR = "bilstm"
MODEL_NAME = "bilstm"

hidden_size = 512
num_layers = 4

# epoch = 10 # Distikaggle
epoch = 6 # PMBF

In [ ]:
from nb2p.stmodel.dnn import BiLSTM

# Instantiate the model
st_model = BiLSTM(
    input_size=MAX_LENGTH, hidden_size=hidden_size, num_layers=num_layers
).to(device)

MODEL_PATH = (
    TRAIN_DIRS.log
    / f"{TRAIN_SETUP_NAME}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}-e{epoch}.pt"
)
print(MODEL_PATH)

st_model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device('cpu')))
st_model.eval()
print("Model loaded")

In [ ]:
segment_result = Trainer(
    st_model,
    num_epochs=300,
    batch_size=128,
    eval_freq=1,
    checkpoint_freq=1,
    learning_rate=1e-3,
    device=device,
    model_write_dir=TEST_DIRS.log,
    report_interval=50,
    window_size=2000,
    model_name=(
        f"{TEST_SETUP_NAME}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}"
    ),
    setup='bce',
).evaluate(
    DNNInput(X=X_test_padded, y=y_test_padded, code=CODE_X, mask=test_mask),
)

In [ ]:
segment_result_dict = {i: s for i, s in enumerate(segment_result)}
print(next(iter(segment_result_dict.values())))

In [ ]:
from tqdm.contrib.concurrent import process_map

MAX_WORKERS = 32

ged_result = process_map(
    dgraph.do_compute_ged,
    segment_result_dict.items(),
    max_workers=MAX_WORKERS,
    chunksize=1,
)
len(ged_result)

In [ ]:
ged_result_valid = [g for g in ged_result if g is not None]
len(ged_result_valid)

In [ ]:
print(sum(ged_result_valid) / len(ged_result_valid))

In [ ]:
fileop.write_json(ged_result, TEST_DIRS.base / f"{TEST_SETUP_NAME}_bilstm-ged.json")